In [13]:
# ------------------------------------->
# Important Packages and Libraries
# ------------------------------------->

import unicodedata
import re 
import os 
import numpy as np
import pytesseract
from PIL import Image 
import cv2
from scipy import ndimage
import matplotlib.pyplot as plt

In [14]:
# Define colors for printing ----------->
YELLOW = '\033[93m'
GREEN = '\033[92m'
ENDC = '\033[0m'
PINK = '\033[95m'
BLUE = '\033[94m'

In [15]:
# ---------------------------------------->
# Image Loading and Normal Preprocessing
# ---------------------------------------->

def load_image(image_path):
    try: 
        img = cv2.imread(image_path)
        if img is None:
            raise FileNotFoundError(f"Image not found at path: {image_path}")
        else:
            return img
    except Exception as e:
        print(f"Error loading image: {e}")
        return None

# Logo Handling --------->
def detect_and_remove_logos(image, min_logo_size=5000, max_logo_size=50000):

    print(PINK + "Detecting and removing logos..." + ENDC)
    
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    logo_mask = np.zeros_like(gray)
    
    for contour in contours:
        area = cv2.contourArea(contour)
        if min_logo_size < area < max_logo_size:
            x, y, w, h = cv2.boundingRect(contour)
            aspect_ratio = w / h
            if 0.3 < aspect_ratio < 3.0:  # Reasonable aspect ratio for logos
                solidity = area / (w * h)
                if solidity > 0.3: 
                    cv2.drawContours(logo_mask, [contour], -1, 255, -1)
    
    if np.sum(logo_mask) > 0:
        image = cv2.inpaint(image, logo_mask, 3, cv2.INPAINT_TELEA)
        print(GREEN + f"Removed {np.sum(logo_mask > 0)} logo pixels" + ENDC)
    else:
        print(GREEN + "No logos detected" + ENDC)
    
    return image

# Image Channel Handling --------------->
def ensure_3_channel(image):
    if len(image.shape) == 2:
        return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    return image

# Ensure Grayscale --------------->
def ensure_grayscale(image):
    if len(image.shape) == 3:
        return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return image

In [16]:
# ---------------------------------------->
# Text Region Recognition and Segmentation
# ---------------------------------------->

def segment_text_regions(image):
    print(PINK + "Segmenting text regions..." + ENDC)
    gray = ensure_grayscale(image)
    masks = []
    
    _, dark_text_mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    masks.append(('dark_text', dark_text_mask))
    
    _, light_text_mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    masks.append(('light_text', light_text_mask))

    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    _, medium_text_mask = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    masks.append(('medium_text', medium_text_mask))

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    
    cleaned_masks = []
    for name, mask in masks:
        cleaned = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel)
        cleaned_masks.append((name, cleaned))
    
    return cleaned_masks, gray

In [17]:
# Handwritten Text Handling --------->
def handle_handwritten_text(image):
    print(PINK + "Processing handwritten text regions..." + ENDC)
    
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    denoised = cv2.bilateralFilter(gray, 9, 75, 75)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(denoised)
    binary = cv2.adaptiveThreshold(enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 35, 15)
    
    return binary

In [18]:
# ---------------------------------------->
#  Advanced Rotation Correction
# ---------------------------------------->

def advanced_rotation_correction(image):
    gray = ensure_grayscale(image)
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLines(edges, 1, np.pi/180, threshold=100)
    
    angles = []
    if lines is not None:
        for i in range(min(20, len(lines))):
            line = lines[i]
            vals = np.asarray(line).ravel()
            if vals.size < 2:
                continue
            rho, theta = vals[0], vals[1]
            angle = np.degrees(theta) - 90
            if -45 <= angle <= 45:
                angles.append(angle)
    
    binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 15, 5)

    if angles:
        median_angle = np.median(angles)
        print(GREEN + f"Correcting rotation by {median_angle:.2f} degrees" + ENDC)
        
        (h, w) = image.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, median_angle, 1.0)
        rotated = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
        return rotated
    
    return image

def resolution(image, scale_factor=2):
    height, width = image.shape[:2]
    new_width = int(width * scale_factor)
    new_height = int(height * scale_factor)
    return cv2.resize(image, (new_width, new_height), interpolation=cv2.INTER_CUBIC)

def remove_noise(image):
    denoised = cv2.bilateralFilter(image, 9, 75, 75)
    kernel = np.ones((1, 1), np.uint8)
    cleaned = cv2.morphologyEx(denoised, cv2.MORPH_CLOSE, kernel)
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, kernel)
    return cleaned


In [ ]:
# -------------------------------->
# Starting Preprocessing Pipeline
# -------------------------------->

def preprocess_pipeline(image_path, enhance=True):
    print(PINK + "Loading image..." + ENDC)
    image = load_image(image_path)
    if image is None:
        return None, None
    print(GREEN + "Image loaded successfully." + ENDC)

    print(PINK + "Logo detection and removal..." + ENDC)
    image = detect_and_remove_logos(image)
    print(GREEN + "Logo processing completed." + ENDC)

    print(PINK + "Advanced rotation correction..." + ENDC)
    image = advanced_rotation_correction(image)
    print(GREEN + "Rotation correction completed." + ENDC)

    print(PINK + "Text region segmentation..." + ENDC)
    text_masks, gray_image = segment_text_regions(image)
    print(GREEN + f"Found {len(text_masks)} text region types" + ENDC)

    print(PINK + "Resolution..." + ENDC)
    if enhance:
        image = resolution(image)
    print(GREEN + "Image resolution successfully." + ENDC)

    print(PINK + "Removing noise..." + ENDC)
    image = remove_noise(image)
    print(GREEN + "Noise removed." + ENDC)

    return image, text_masks


def safe_bitwise(image, mask):
    # Ensure image is 3-channel and mask is single-channel
    if len(image.shape) == 3:
        image_3ch = image
    else:
        image_3ch = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    
    # Ensure mask is the same size as image
    if mask.shape[:2] != image_3ch.shape[:2]:
        mask = cv2.resize(mask, (image_3ch.shape[1], image_3ch.shape[0]))
    
    # Ensure mask is uint8
    mask = mask.astype(np.uint8)
    
    return cv2.bitwise_and(image_3ch, image_3ch, mask=mask)

def region_specific_ocr(image, text_masks):
    print(PINK + "Performing region-specific OCR..." + ENDC)
    
    all_results = []
    
    for region_name, mask in text_masks:
        print(BLUE + f"Processing {region_name}..." + ENDC)
        
        # Apply mask to original image safely
        try:
            masked_image = safe_bitwise(image, mask)

            if len(masked_image.shape) == 3:
                masked_gray = cv2.cvtColor(masked_image, cv2.COLOR_BGR2GRAY)
            else:
                masked_gray = masked_image
                
        except Exception as e:
            print(f"Error processing {region_name}: {e}")
            continue
        
        if 'dark_text' in region_name:
            # Standard printed text
            configs = [
                r'--oem 3 --psm 6 -l khm',
                r'--oem 3 --psm 4 -l khm',
                r'-l khm+eng --oem 3 --psm 6'
            ]
        else:
            # Handwritten or special text
            configs = [
                r'--oem 3 --psm 8 -l khm',  # Single word
                r'--oem 3 --psm 7 -l khm',  # Single line
                r'--oem 3 --psm 13 -l khm',  # Raw line
                r'-l khm+eng --oem 3 --psm 6'
            ]
        
        region_results = []
        for config in configs:
            try:
                text = pytesseract.image_to_string(masked_gray, config=config)
                confidence_data = pytesseract.image_to_data(masked_gray, config=config, output_type=pytesseract.Output.DICT)
                
                # Calculate average confidence
                confidences = [int(c) for c in confidence_data['conf'] if int(c) > 0]
                avg_confidence = np.mean(confidences) if confidences else 0
                
                if text.strip():  # Only consider non-empty results
                    region_results.append({
                        'config': config,
                        'text': text.strip(),
                        'confidence': avg_confidence
                    })
            except Exception as e:
                print(f"OCR error with config {config}: {e}")
                continue
        
        # Select best result for this region
        if region_results:
            best_region = max(region_results, key=lambda x: x['confidence'])
            all_results.append({
                'region': region_name,
                'text': best_region['text'],
                'confidence': best_region['confidence']
            })
            print(f"  {region_name}: {len(best_region['text'])} chars, confidence: {best_region['confidence']:.2f}")

    combined_text = ' '.join([r['text'] for r in all_results if r['text']])
    overall_confidence = np.mean([r['confidence'] for r in all_results]) if all_results else 0
    
    print(GREEN + f"Combined {len(all_results)} regions, confidence: {overall_confidence:.2f}" + ENDC)
    
    return {
        'combined_text': combined_text,
        'region_results': all_results,
        'confidence': overall_confidence
    }


In [20]:
#------------------------------->
# Multi-Pass OCR Strategy
#-------------------------------> 

def multi_pass_ocr(image, text_masks):

    print(YELLOW + "Starting multi-pass OCR..." + ENDC)

    # Pass 1: Region-specific OCR (returns a dict)
    region_results = region_specific_ocr(image, text_masks)
    # normalize to get the list of region dicts and summary values
    region_list = region_results.get('region_results', []) if isinstance(region_results, dict) else region_results
    region_combined_text = region_results.get('combined_text', '') if isinstance(region_results, dict) else ' '.join([r.get('text','') for r in region_list])
    region_confidence = region_results.get('confidence', 0) if isinstance(region_results, dict) else (np.mean([r.get('confidence',0) for r in region_list]) if region_list else 0)

    whole_image_configs = [
        r'--oem 3 --psm 6 -l khm',  # Standard
        r'--oem 3 --psm 4 -l khm',  # Multiple blocks
        r'--oem 3 --psm 11 -l khm', # Sparse text
        r'-l khm+eng --oem 3 --psm 6'
    ]

    whole_image_results = []
    for config in whole_image_configs:
        try:
            if len(image.shape) == 3:
                gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            else:
                gray_image = image

            text = pytesseract.image_to_string(gray_image, config=config)
            confidence_data = pytesseract.image_to_data(gray_image, config=config, output_type=pytesseract.Output.DICT)

            confs = []
            for c in confidence_data.get('conf', []):
                try:
                    cv = int(float(c))
                    if cv > 0:
                        confs.append(cv)
                except Exception:
                    continue

            avg_confidence = float(np.mean(confs)) if confs else 0

            if text.strip():
                whole_image_results.append({
                    'config': config,
                    'text': text.strip(),
                    'confidence': avg_confidence
                })
        except Exception as e:
            print(f"Whole image OCR error: {e}")
            continue

    # Select best whole image result
    best_whole = max(whole_image_results, key=lambda x: x['confidence']) if whole_image_results else {'text': '', 'confidence': 0}

    # Combine region-based and whole-image results
    all_texts = [region_combined_text, best_whole['text']]
    all_confidences = [region_confidence, best_whole['confidence']]

    # Use the result with highest confidence (guard for empty)
    if not any(all_confidences):
        final_text = ''
        final_confidence = 0
        result_type = "none"
    else:
        best_idx = int(np.argmax(all_confidences))
        final_text = all_texts[best_idx]
        final_confidence = float(all_confidences[best_idx])
        result_type = "region_based" if best_idx == 0 else "whole_image"

    print(GREEN + f"Selected {result_type} result with confidence: {final_confidence:.2f}" + ENDC)

    return {
        'text': final_text,
        'confidence': final_confidence,
        'region_results': region_list,
        'whole_image_results': whole_image_results
    }

In [21]:
#---------------------------------------------------------->
#  Handling the Error Manually in Khmer Text
#  Further improvements can be made by expanding the dictionary and rules based on more comprehensive linguistic analysis.
#----------------------------------------------------------->

def normalize_khmer_unicode(text):
    normalized = unicodedata.normalize('NFC', text)
    error_corrections = {
        'ាឹ': 'ា',
        'ិះ': 'ិ',
        'ុះ': 'ុ',
        '៉ា': 'ា',
    }
    for error, correction in error_corrections.items():
        normalized = normalized.replace(error, correction)
    return normalized

def correct_khmer_spacing(text):
    khmer_pattern = r'([\u1780-\u17FF]+)\s+([\u1780-\u17FF]+)'
    corrected = re.sub(khmer_pattern, r'\1\2', text)
    return corrected.strip()

def numbers_to_khmer(text):
    arabic_digits = '0123456789'
    khmer_digits = '០១២៣៤៥៦៧៨៩'
    translation_table = str.maketrans(arabic_digits, khmer_digits)
    return text.translate(translation_table)

def expand_khmer_abbreviations(text):
    abbreviations = {
        'គ.ស.': 'គ្រិស្តសករាជ',
        'ម.រ.': 'មុនគ្រិស្តសករាជ',
        'រ.ដ.': 'រដ្ឋាភិបាល',
        'ឯ.អ.': 'ឯកអគ្គ',
        'អ.ដ.': 'អនុដ្ឋាន',
        'ស.រ.': 'សាធារណរដ្ឋ',
    }
    for abbr, full in abbreviations.items():
        text = text.replace(abbr, full)
    return text

def spell_check_khmer(text, custom_dict=None):
    common_errors = {
        'មហាុ': 'មហា',
        'ប្រទេសជ': 'ប្រទេស',
        'កមពុជ': 'កម្ពុជា',
        'បរជាតិយ': 'ប្រជាជាតិ',
        'សកលវិទ្យាលយ': 'សកលវិទ្យាល័យ',
        'អង្គរវត្ត្': 'អង្គរវត្ត',
        'បាំបាំ': 'ប៉ាប៉ា',
    }
    for error, correction in common_errors.items():
        text = text.replace(error, correction)
    return text

In [22]:
# =====================================>
# Starting Post-processing Pipeline 
# =====================================>

def postprocess_pipeline(text):
    
    # Original normalization steps
    text = normalize_khmer_unicode(text)
    text = correct_khmer_spacing(text)
    text = numbers_to_khmer(text)
    text = expand_khmer_abbreviations(text)
    text = spell_check_khmer(text)
    
    # Additional cleaning
    text = re.sub(r'\s+', ' ', text)  # Normalize whitespace
    text = text.strip()
    
    return text

In [ ]:
#----------------------------------->
# Main workflow of preprocessing
#----------------------------------->

def process_document(image_path, output_dir="/media/chhaythean/Drive D/Ai-Edu/data/images/output"):
    print(YELLOW + "    Starting Enhanced Document Processing    " + ENDC)
    os.makedirs(output_dir, exist_ok=True)

    try:
        # preprocessing
        preprocessed_image, text_masks = preprocess_pipeline(image_path)
        if preprocessed_image is None:
            return None

        # Save preprocessed image
        preprocessed_path = os.path.join(output_dir, "preprocessed.png")
        cv2.imwrite(preprocessed_path, preprocessed_image)

        # OCR with region processing
        print(YELLOW + "Performing  multi-pass OCR..." + ENDC)
        ocr_result = multi_pass_ocr(preprocessed_image, text_masks)
        print(GREEN + f"OCR completed with confidence: {ocr_result['confidence']:.2f}" + ENDC)

        # Post-processing
        print(YELLOW + "Post-processing..." + ENDC)
        cleaned_text = postprocess_pipeline(ocr_result['text'])

        # Save detailed results
        results = {
            'original_text': ocr_result['text'],
            'cleaned_text': cleaned_text,
            'preprocessed_image': preprocessed_path,
            'confidence': ocr_result['confidence'],
            'region_results': ocr_result.get('region_results', []),
            'whole_image_results': ocr_result.get('whole_image_results', [])
        }

        # Save output
        with open(os.path.join(output_dir, "extracted_text.txt"), "w", encoding="utf-8") as f:
            f.write(cleaned_text)

        # Save region analysis
        with open(os.path.join(output_dir, "region_analysis.txt"), "w", encoding="utf-8") as f:
            for region in results.get('region_results', []):
                f.write(f"Region: {region['region']}\n")
                f.write(f"Confidence: {region['confidence']:.2f}\n")
                f.write(f"Text: {region['text']}\n")
                f.write("-" * 50 + "\n")

        print(GREEN + "             Processing Completed           " + ENDC)
        print(GREEN + f"Extracted {len(cleaned_text)} characters with confidence {results['confidence']:.2f}" + ENDC)

        return results

    except Exception as e:
        print(f"Error in processing: {e}")
        import traceback
        traceback.print_exc()
        return None

In [24]:
#--------------------------------->
# Testing the complete pipeline
#--------------------------------->

image_test = process_document(r"/media/chhaythean/Drive D/Ai-Edu/data/images/doc2.jpg", output_dir="/media/chhaythean/Drive D/Ai-Edu/data/images/output")

    Starting Enhanced Document Processing    
Loading image...
Image loaded successfully.
Logo detection and removal...
Detecting and removing logos...
Removed 64914 logo pixels
Logo processing completed.
Advanced rotation correction...
Correcting rotation by 0.00 degrees
Rotation correction completed.
Text region segmentation...
Segmenting text regions...
Found 3 text region types
Resolution...
Image resolution successfully.
Removing noise...
Noise removed.
Performing  multi-pass OCR...
Starting multi-pass OCR...
Performing region-specific OCR...
Processing dark_text...
  dark_text: 2190 chars, confidence: 63.87
Processing light_text...
  light_text: 2144 chars, confidence: 72.31
Processing medium_text...
  medium_text: 2100 chars, confidence: 64.92
Combined 3 regions, confidence: 67.03
Selected whole_image result with confidence: 77.58
OCR completed with confidence: 77.58
Post-processing...
             Processing Completed           
Extracted 2099 characters with confidence 77.58
